In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('../data/secom.data', sep=' ', header=None)
labels = pd.read_csv('../data/secom_labels.data', sep=' ', header=None)

# Basic info
print(f"Dataset shape: {df.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Total missing values: {df.isnull().sum().sum()}")
print(f"Missing percentage: {df.isnull().mean().mean()*100:.1f}%")

Dataset shape: (1567, 590)
Labels shape: (1567, 2)
Total missing values: 41951
Missing percentage: 4.5%


## Dataset Overview

The SECOM dataset contains sensor readings from a semiconductor manufacturing process.

- **1,567 production runs** recorded
- **590 process sensors** monitored per run
- **4.5% missing data** — typical of real industrial sensor systems where readings 
  occasionally fail or fall outside measurement range
- Labels indicate pass/fail outcome for each production run

Missing values will be handled by column-wise median imputation — a robust approach 
for sensor data where missingness is random rather than systematic.

# Rename label columns for clarity
labels.columns = ['outcome', 'timestamp']

# Check pass/fail balance
outcome_counts = labels['outcome'].value_counts()
print("Pass/Fail breakdown:")
print(f"  Pass (-1): {outcome_counts.get(-1, 0)} runs ({outcome_counts.get(-1, 0)/len(labels)*100:.1f}%)")
print(f"  Fail  (1): {outcome_counts.get(1, 0)} runs ({outcome_counts.get(1, 0)/len(labels)*100:.1f}%)")

In [2]:
# Rename label columns for clarity
labels.columns = ['outcome', 'timestamp']

# Check pass/fail balance
outcome_counts = labels['outcome'].value_counts()
print("Pass/Fail breakdown:")
print(f"  Pass (-1): {outcome_counts.get(-1, 0)} runs ({outcome_counts.get(-1, 0)/len(labels)*100:.1f}%)")
print(f"  Fail  (1): {outcome_counts.get(1, 0)} runs ({outcome_counts.get(1, 0)/len(labels)*100:.1f}%)")

Pass/Fail breakdown:
  Pass (-1): 1463 runs (93.4%)
  Fail  (1): 104 runs (6.6%)


In [3]:
# Handle missing values using column-wise median imputation
# Median is preferred over mean for sensor data — robust to outliers
df_clean = df.fillna(df.median())

# Remove columns where more than 50% of values were missing
# These columns carry too little information to be reliable
missing_ratio = df.isnull().mean()
cols_to_drop = missing_ratio[missing_ratio > 0.5].index
df_clean = df_clean.drop(columns=cols_to_drop)

print(f"Columns removed due to >50% missing: {len(cols_to_drop)}")
print(f"Remaining features: {df_clean.shape[1]}")
print(f"Missing values remaining: {df_clean.isnull().sum().sum()}")
print(f"Dataset is clean: {df_clean.isnull().sum().sum() == 0}")

Columns removed due to >50% missing: 28
Remaining features: 562
Missing values remaining: 0
Dataset is clean: True


In [4]:
import os

# Save cleaned dataset to outputs folder
df_clean.to_csv('../outputs/secom_clean.csv', index=False)
labels.to_csv('../outputs/secom_labels_clean.csv', index=False)

print(f"Clean dataset saved: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")
print(f"Labels saved: {labels.shape[0]} rows")
print("Files written to /outputs/")

Clean dataset saved: 1567 rows x 562 columns
Labels saved: 1567 rows
Files written to /outputs/
